# World Cup 2026 EDA and Winner Forecast

**Goal:** Analyze available FIFA World Cup 2026 data through July 15, reconstruct the remaining bracket, simulate the champion probabilities, and generate a polished HTML report.

**Dataset:** `654a303c` (`fifawc26_tilljul15`)

**Important caveat:** This is a data-driven probabilistic forecast based only on the provided dataset. It does not include live injuries, betting markets, external squad news, or tactical updates.

In [2]:
# Cell: setup imports and output paths
from pathlib import Path
import json
import math
import hashlib
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import dataclaw_data

DATASET_ID = "654a303c"
# Notebook runs from its own folder, so outputs are written beside this notebook.
OUT_DIR = Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(20260715)

TABLES = {
    "matches": "file_fifawc26_15jul_matches_csv",
    "matches_detailed": "file_fifawc26_15jul_matches_detailed_csv",
    "teams": "file_fifawc26_15jul_teams_csv",
    "team_stats": "file_fifawc26_15jul_match_team_stats_csv",
    "player_stats": "file_fifawc26_15jul_player_stats_csv",
    "squads": "file_fifawc26_15jul_squads_and_players_csv",
    "stages": "file_fifawc26_15jul_tournament_stages_csv",
}
print(f"Output directory: {OUT_DIR.resolve()}")

Output directory: /Users/nandinimathan/.dataclaw/workspaces/37635ff9-9679-4194-ade3-b26f4e3e1175/world_cup_analysis


In [3]:
# Cell: load core tables and audit inventory
matches = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["matches"])
md = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["matches_detailed"])
teams = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["teams"])
team_stats = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["team_stats"])
player_stats = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["player_stats"])
squads = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["squads"])
stages = dataclaw_data.get_dataframe(DATASET_ID, table_name=TABLES["stages"])

inventory = pd.DataFrame([
    {"table": "matches", "rows": len(matches), "columns": matches.shape[1]},
    {"table": "matches_detailed", "rows": len(md), "columns": md.shape[1]},
    {"table": "teams", "rows": len(teams), "columns": teams.shape[1]},
    {"table": "team_stats", "rows": len(team_stats), "columns": team_stats.shape[1]},
    {"table": "player_stats", "rows": len(player_stats), "columns": player_stats.shape[1]},
    {"table": "squads", "rows": len(squads), "columns": squads.shape[1]},
    {"table": "stages", "rows": len(stages), "columns": stages.shape[1]},
])

status_stage = md.groupby(["stage_name", "status"], dropna=False).size().reset_index(name="matches")
missing_core = md[["home_team_name", "away_team_name", "home_score", "away_score", "home_xg", "away_xg"]].isna().mean().reset_index()
missing_core.columns = ["field", "missing_rate"]

inventory.to_csv(OUT_DIR / "data_inventory.csv", index=False)
status_stage.to_csv(OUT_DIR / "match_status_by_stage.csv", index=False)
missing_core.to_csv(OUT_DIR / "core_missingness.csv", index=False)

print("Inventory")
display(inventory)
print("\nMatch status by stage")
display(status_stage)
print("\nCore missingness")
display(missing_core)

Inventory


,table,rows,columns
0,matches,104,17
1,matches_detailed,104,23
2,teams,48,8
3,team_stats,202,12
4,player_stats,1248,21
5,squads,1248,10
6,stages,7,3



Match status by stage


,stage_name,status,matches
0,Final,Scheduled,1
1,Group Stage,Completed,72
2,Quarter-finals,Completed,4
3,Round of 16,Completed,8
4,Round of 32,Completed,16
5,Semi-finals,Completed,1
6,Semi-finals,Scheduled,1
7,Third-place match,Scheduled,1



Core missingness


,field,missing_rate
0,home_team_name,0.019231
1,away_team_name,0.019231
2,home_score,0.028846
3,away_score,0.028846
4,home_xg,0.028846
5,away_xg,0.028846


In [4]:
# Cell: bracket and scheduled match audit
scheduled = md[md["status"].eq("Scheduled")].copy()
knockout = md[md["stage_name"].isin(["Round of 32", "Round of 16", "Quarter-finals", "Semi-finals", "Third-place match", "Final"])].copy()

def match_winner(row):
    if pd.isna(row["home_score"]) or pd.isna(row["away_score"]):
        return None
    if row["home_score"] > row["away_score"]:
        return row["home_team_name"]
    if row["away_score"] > row["home_score"]:
        return row["away_team_name"]
    # penalties when regular/ET draw
    if pd.notna(row.get("home_penalty_score")) and pd.notna(row.get("away_penalty_score")):
        return row["home_team_name"] if row["home_penalty_score"] > row["away_penalty_score"] else row["away_team_name"]
    return "Draw/Unknown"

knockout["winner"] = knockout.apply(match_winner, axis=1)
completed_kos = knockout[knockout["status"].eq("Completed")]

# Teams in completed semi-finals are confirmed finalists/third-place participants depending winner/loser.
semi_completed = completed_kos[completed_kos["stage_name"].eq("Semi-finals")].copy()
semi_scheduled = scheduled[scheduled["stage_name"].eq("Semi-finals")].copy()
final_scheduled = scheduled[scheduled["stage_name"].eq("Final")].copy()
third_scheduled = scheduled[scheduled["stage_name"].eq("Third-place match")].copy()

scheduled_display = scheduled[["match_id", "date", "stage_name", "home_team_name", "away_team_name", "stadium_name", "city", "country"]]
completed_ko_display = completed_kos[["match_id", "stage_name", "home_team_name", "away_team_name", "home_score", "away_score", "home_penalty_score", "away_penalty_score", "winner"]].tail(16)

scheduled_display.to_csv(OUT_DIR / "scheduled_remaining_matches.csv", index=False)
completed_ko_display.to_csv(OUT_DIR / "recent_knockout_results.csv", index=False)

print("Remaining scheduled fixtures")
display(scheduled_display)
print("\nRecent knockout results")
display(completed_ko_display)

Remaining scheduled fixtures


,match_id,date,stage_name,home_team_name,away_team_name,stadium_name,city,country
101,102,2026-07-15,Semi-finals,England,Argentina,Atlanta Stadium (Mercedes-Benz Stadium),Atlanta,USA
102,103,2026-07-18,Third-place match,None,None,Miami Stadium (Hard Rock Stadium),Miami Gardens,USA
103,104,2026-07-19,Final,None,None,New York New Jersey Stadium (MetLife Stadium),East Rutherford,USA



Recent knockout results


,match_id,stage_name,home_team_name,away_team_name,home_score,away_score,home_penalty_score,away_penalty_score,winner
85,86,Round of 32,Australia,Egypt,1.0,1.0,2.0,4.0,Egypt
86,87,Round of 32,Argentina,Cabo Verde,3.0,2.0,NaN,NaN,Argentina
87,88,Round of 32,Colombia,Ghana,1.0,0.0,NaN,NaN,Colombia
88,89,Round of 16,Paraguay,France,0.0,1.0,NaN,NaN,France
89,90,Round of 16,Canada,Morocco,0.0,3.0,NaN,NaN,Morocco
90,91,Round of 16,Brazil,Norway,1.0,2.0,NaN,NaN,Norway
91,92,Round of 16,Mexico,England,2.0,3.0,NaN,NaN,England
92,93,Round of 16,Portugal,Spain,0.0,1.0,NaN,NaN,Spain
93,94,Round of 16,USA,Belgium,1.0,4.0,NaN,NaN,Belgium
94,95,Round of 16,Argentina,Egypt,3.0,2.0,NaN,NaN,Argentina


In [5]:
# Cell: team-level EDA feature construction
completed = md[md["status"].eq("Completed")].copy()

# Convert match-level data into team-match rows.
home_rows = completed[["match_id", "date", "stage_name", "home_team_name", "away_team_name", "home_score", "away_score", "home_xg", "away_xg"]].copy()
home_rows.columns = ["match_id", "date", "stage_name", "team_name", "opponent", "goals_for", "goals_against", "xg_for", "xg_against"]
home_rows["is_home"] = 1

away_rows = completed[["match_id", "date", "stage_name", "away_team_name", "home_team_name", "away_score", "home_score", "away_xg", "home_xg"]].copy()
away_rows.columns = ["match_id", "date", "stage_name", "team_name", "opponent", "goals_for", "goals_against", "xg_for", "xg_against"]
away_rows["is_home"] = 0

team_match = pd.concat([home_rows, away_rows], ignore_index=True)
team_match["goal_diff"] = team_match["goals_for"] - team_match["goals_against"]
team_match["xg_diff"] = team_match["xg_for"] - team_match["xg_against"]
team_match["won"] = (team_match["goal_diff"] > 0).astype(int)
team_match["draw"] = (team_match["goal_diff"] == 0).astype(int)
team_match["points"] = np.where(team_match["goal_diff"] > 0, 3, np.where(team_match["goal_diff"] == 0, 1, 0))
team_match["is_knockout"] = team_match["stage_name"].ne("Group Stage")

team_perf = team_match.groupby("team_name").agg(
    matches_played=("match_id", "count"),
    wins=("won", "sum"),
    draws=("draw", "sum"),
    points=("points", "sum"),
    goals_for=("goals_for", "sum"),
    goals_against=("goals_against", "sum"),
    goal_diff=("goal_diff", "sum"),
    xg_for=("xg_for", "sum"),
    xg_against=("xg_against", "sum"),
    xg_diff=("xg_diff", "sum"),
    avg_xg_diff=("xg_diff", "mean"),
    knockout_matches=("is_knockout", "sum"),
).reset_index()
team_perf["points_per_match"] = team_perf["points"] / team_perf["matches_played"]
team_perf["goals_per_match"] = team_perf["goals_for"] / team_perf["matches_played"]
team_perf["xg_diff_per_match"] = team_perf["xg_diff"] / team_perf["matches_played"]

team_features = teams.merge(team_perf, on="team_name", how="left").fillna({
    "matches_played": 0, "wins": 0, "draws": 0, "points": 0, "goals_for": 0,
    "goals_against": 0, "goal_diff": 0, "xg_for": 0, "xg_against": 0,
    "xg_diff": 0, "avg_xg_diff": 0, "knockout_matches": 0, "points_per_match": 0,
    "goals_per_match": 0, "xg_diff_per_match": 0,
})

# Shrink tournament performance toward zero because match counts are small.
team_features["form_weight"] = team_features["matches_played"] / (team_features["matches_played"] + 5)
team_features["shrunk_xg_diff_pm"] = team_features["xg_diff_per_match"] * team_features["form_weight"]
team_features["shrunk_goal_diff_pm"] = (team_features["goal_diff"] / team_features["matches_played"].replace(0, np.nan)).fillna(0) * team_features["form_weight"]
team_features["rank_score"] = -team_features["fifa_ranking_pre_tournament"]

# Composite is standardized mix of Elo and shrunk tournament form.
for col in ["elo_rating", "shrunk_xg_diff_pm", "shrunk_goal_diff_pm", "rank_score"]:
    team_features[f"z_{col}"] = (team_features[col] - team_features[col].mean()) / team_features[col].std(ddof=0)
team_features["forecast_rating"] = (
    0.55 * team_features["z_elo_rating"] +
    0.25 * team_features["z_shrunk_xg_diff_pm"] +
    0.15 * team_features["z_shrunk_goal_diff_pm"] +
    0.05 * team_features["z_rank_score"]
)

remaining_teams = ["Spain", "England", "Argentina", "France"]
remaining_feature_table = team_features[team_features["team_name"].isin(remaining_teams)].sort_values("forecast_rating", ascending=False)
top_feature_table = team_features.sort_values("forecast_rating", ascending=False).head(12)

team_match.to_csv(OUT_DIR / "team_match_rows.csv", index=False)
team_features.to_csv(OUT_DIR / "team_features.csv", index=False)
remaining_feature_table.to_csv(OUT_DIR / "remaining_team_features.csv", index=False)
top_feature_table.to_csv(OUT_DIR / "top_team_features.csv", index=False)

display_cols = ["team_name", "elo_rating", "fifa_ranking_pre_tournament", "matches_played", "goals_for", "goals_against", "goal_diff", "xg_diff", "xg_diff_per_match", "forecast_rating"]
print("Remaining contenders feature table")
display(remaining_feature_table[display_cols].round(3))
print("\nTop 12 composite team ratings")
display(top_feature_table[display_cols].round(3))

Remaining contenders feature table


,team_name,elo_rating,fifa_ranking_pre_tournament,matches_played,goals_for,goals_against,goal_diff,xg_diff,xg_diff_per_match,forecast_rating
28,Spain,2120,2,7,13.0,1.0,12.0,12.02,1.717,2.031
36,Argentina,2150,1,6,17.0,6.0,11.0,8.58,1.430,2.010
32,France,2100,3,7,16.0,4.0,12.0,8.45,1.207,1.790
44,England,2050,4,6,13.0,6.0,7.0,3.41,0.568,1.297



Top 12 composite team ratings


,team_name,elo_rating,fifa_ranking_pre_tournament,matches_played,goals_for,goals_against,goal_diff,xg_diff,xg_diff_per_match,forecast_rating
28,Spain,2120,2,7,13.0,1.0,12.0,12.02,1.717,2.031
36,Argentina,2150,1,6,17.0,6.0,11.0,8.58,1.430,2.010
32,France,2100,3,7,16.0,4.0,12.0,8.45,1.207,1.790
8,Brazil,2030,6,5,10.0,4.0,6.0,5.53,1.106,1.353
44,England,2050,4,6,13.0,6.0,7.0,3.41,0.568,1.297
16,Germany,1970,10,4,11.0,5.0,6.0,6.08,1.520,1.224
43,Colombia,1990,13,5,5.0,1.0,4.0,5.32,1.064,1.136
40,Portugal,2010,5,5,8.0,3.0,5.0,2.64,0.528,1.095
20,Netherlands,1980,8,4,11.0,5.0,6.0,3.06,0.765,1.071
24,Belgium,1950,9,6,14.0,7.0,7.0,1.99,0.332,0.867


In [6]:
# Cell: EDA visualizations for scoring and contender strength
stage_order = ["Group Stage", "Round of 32", "Round of 16", "Quarter-finals", "Semi-finals"]
stage_scoring = completed.assign(total_goals=completed["home_score"] + completed["away_score"], total_xg=completed["home_xg"] + completed["away_xg"]).groupby("stage_name").agg(
    matches=("match_id", "count"),
    goals_per_match=("total_goals", "mean"),
    xg_per_match=("total_xg", "mean"),
).reindex(stage_order).dropna().reset_index()

fig_stage = go.Figure()
fig_stage.add_trace(go.Bar(x=stage_scoring["stage_name"], y=stage_scoring["goals_per_match"], name="Goals per match"))
fig_stage.add_trace(go.Scatter(x=stage_scoring["stage_name"], y=stage_scoring["xg_per_match"], mode="lines+markers", name="xG per match"))
fig_stage.update_layout(title="Scoring and chance volume by stage", xaxis_title="Tournament stage", yaxis_title="Per-match average", barmode="group")
fig_stage.show()

contenders = remaining_feature_table.copy()
fig_contenders = px.bar(
    contenders.sort_values("forecast_rating"),
    x="forecast_rating", y="team_name", orientation="h",
    color="xg_diff_per_match",
    hover_data=["elo_rating", "fifa_ranking_pre_tournament", "goal_diff", "xg_diff"],
    title="Composite forecast rating for remaining contenders",
    labels={"forecast_rating":"Composite forecast rating", "team_name":"Team", "xg_diff_per_match":"xG diff / match"}
)
fig_contenders.show()

stage_scoring.to_csv(OUT_DIR / "stage_scoring_summary.csv", index=False)
fig_stage.write_html(OUT_DIR / "stage_scoring_chart.html", include_plotlyjs="cdn")
fig_contenders.write_html(OUT_DIR / "contender_rating_chart.html", include_plotlyjs="cdn")

stage_scoring.round(3)

,stage_name,matches,goals_per_match,xg_per_match
0,Group Stage,72,2.986,2.658
1,Round of 32,16,2.625,2.589
2,Round of 16,8,2.875,2.832
3,Quarter-finals,4,3.000,2.485
4,Semi-finals,1,2.000,1.930


In [7]:
# Cell: forecast model, baseline checks, and champion simulation
import mlflow

rating_lookup = team_features.set_index("team_name")["forecast_rating"].to_dict()
elo_lookup = team_features.set_index("team_name")["elo_rating"].to_dict()

# Convert composite rating differences into match win probabilities.
# Scale chosen so a 1.0 composite-rating edge is meaningful but not deterministic.
def logistic(x):
    return 1 / (1 + np.exp(-x))

def p_win_composite(team_a, team_b, scale=1.05):
    return float(logistic((rating_lookup[team_a] - rating_lookup[team_b]) * scale))

def p_win_elo(team_a, team_b):
    # Standard Elo expectation; neutral-site assumption.
    return float(1 / (1 + 10 ** ((elo_lookup[team_b] - elo_lookup[team_a]) / 400)))

matchup_probs = pd.DataFrame([
    {"match":"Semi-final", "team_a":"Argentina", "team_b":"England", "composite_p_team_a":p_win_composite("Argentina", "England"), "elo_p_team_a":p_win_elo("Argentina", "England")},
    {"match":"Potential final", "team_a":"Spain", "team_b":"Argentina", "composite_p_team_a":p_win_composite("Spain", "Argentina"), "elo_p_team_a":p_win_elo("Spain", "Argentina")},
    {"match":"Potential final", "team_a":"Spain", "team_b":"England", "composite_p_team_a":p_win_composite("Spain", "England"), "elo_p_team_a":p_win_elo("Spain", "England")},
])

N = 100_000
champions = []
paths = []
for i in range(N):
    # Remaining SF: Argentina vs England (source lists England home, Argentina away; neutral model ignores order)
    arg_beats_eng = np.random.random() < p_win_composite("Argentina", "England")
    finalist = "Argentina" if arg_beats_eng else "England"
    sf_loser = "England" if arg_beats_eng else "Argentina"
    spain_wins_final = np.random.random() < p_win_composite("Spain", finalist)
    champion = "Spain" if spain_wins_final else finalist
    champions.append(champion)
    paths.append({"sf_winner": finalist, "sf_loser": sf_loser, "finalist_vs_spain": finalist, "champion": champion})

sim_paths = pd.DataFrame(paths)
champion_probs = sim_paths["champion"].value_counts(normalize=True).rename_axis("team_name").reset_index(name="champion_probability")
champion_probs["simulations"] = N
champion_probs = champion_probs.merge(team_features[["team_name", "elo_rating", "fifa_ranking_pre_tournament", "forecast_rating", "xg_diff_per_match", "goal_diff"]], on="team_name", how="left")
champion_probs = champion_probs.sort_values("champion_probability", ascending=False)

# Baseline Elo-only path probabilities are analytic for the same bracket.
p_arg_sf_elo = p_win_elo("Argentina", "England")
p_spain_arg_elo = p_win_elo("Spain", "Argentina")
p_spain_eng_elo = p_win_elo("Spain", "England")
elo_baseline = pd.DataFrame([
    {"team_name":"Spain", "champion_probability": p_arg_sf_elo*p_spain_arg_elo + (1-p_arg_sf_elo)*p_spain_eng_elo, "method":"Elo-only baseline"},
    {"team_name":"Argentina", "champion_probability": p_arg_sf_elo*(1-p_spain_arg_elo), "method":"Elo-only baseline"},
    {"team_name":"England", "champion_probability": (1-p_arg_sf_elo)*(1-p_spain_eng_elo), "method":"Elo-only baseline"},
])

composite_probs = champion_probs[["team_name", "champion_probability"]].assign(method="Composite simulation")
prob_comparison = pd.concat([composite_probs, elo_baseline], ignore_index=True)

# Simple uncertainty interval: binomial Monte Carlo interval around simulated champion share.
champion_probs["mc_se"] = np.sqrt(champion_probs["champion_probability"] * (1 - champion_probs["champion_probability"]) / N)
champion_probs["ci95_low"] = champion_probs["champion_probability"] - 1.96 * champion_probs["mc_se"]
champion_probs["ci95_high"] = champion_probs["champion_probability"] + 1.96 * champion_probs["mc_se"]

matchup_probs.to_csv(OUT_DIR / "matchup_probabilities.csv", index=False)
champion_probs.to_csv(OUT_DIR / "champion_probabilities.csv", index=False)
prob_comparison.to_csv(OUT_DIR / "champion_probability_baseline_comparison.csv", index=False)

# Log a lightweight MLflow run for the predictive analysis.
with mlflow.start_run(run_name="World Cup champion simulation"):
    mlflow.set_tag("analysis_type", "path_dependent_forecast")
    mlflow.set_tag("dataset_id", DATASET_ID)
    mlflow.set_tag("model_type", "heuristic_composite_rating_simulation")
    mlflow.log_param("simulations", N)
    mlflow.log_param("composite_scale", 1.05)
    mlflow.log_param("elo_weight", 0.55)
    mlflow.log_param("xg_form_weight", 0.25)
    mlflow.log_param("goal_form_weight", 0.15)
    mlflow.log_param("rank_weight", 0.05)
    for _, row in champion_probs.iterrows():
        mlflow.log_metric(f"champion_probability_{row['team_name'].lower()}", float(row["champion_probability"]))
    mlflow.log_artifact(str(OUT_DIR / "champion_probabilities.csv"))
    mlflow.log_artifact(str(OUT_DIR / "matchup_probabilities.csv"))

print("Match probabilities")
display(matchup_probs.round(3))
print("\nChampion probabilities")
display(champion_probs[["team_name", "champion_probability", "ci95_low", "ci95_high", "forecast_rating", "elo_rating", "xg_diff_per_match"]].round(4))
print("\nBaseline comparison")
display(prob_comparison.pivot(index="team_name", columns="method", values="champion_probability").round(4))

Match probabilities


,match,team_a,team_b,composite_p_team_a,elo_p_team_a
0,Semi-final,Argentina,England,0.679,0.640
1,Potential final,Spain,Argentina,0.505,0.457
2,Potential final,Spain,England,0.684,0.599



Champion probabilities


,team_name,champion_probability,ci95_low,ci95_high,forecast_rating,elo_rating,xg_diff_per_match
0,Spain,0.5594,0.5564,0.5625,2.0310,2120,1.7171
1,Argentina,0.3381,0.3352,0.3410,2.0104,2150,1.4300
2,England,0.1025,0.1006,0.1044,1.2971,2050,0.5683



Baseline comparison


method,Composite simulation,Elo-only baseline
team_name,,
Argentina,0.3381,0.3476
England,0.1025,0.1442
Spain,0.5594,0.5082


In [8]:
# Cell: forecast visuals
prob_plot = champion_probs.copy()
prob_plot["Champion probability"] = prob_plot["champion_probability"] * 100
prob_plot["Lower 95% MC"] = prob_plot["ci95_low"] * 100
prob_plot["Upper 95% MC"] = prob_plot["ci95_high"] * 100

fig_champ = px.bar(
    prob_plot.sort_values("Champion probability"),
    x="Champion probability", y="team_name", orientation="h",
    text=prob_plot.sort_values("Champion probability")["Champion probability"].map(lambda x: f"{x:.1f}%"),
    hover_data=["forecast_rating", "elo_rating", "xg_diff_per_match"],
    title="Simulated World Cup champion probabilities",
    labels={"team_name":"Team", "Champion probability":"Champion probability (%)"}
)
fig_champ.update_traces(textposition="outside")
fig_champ.update_layout(xaxis_range=[0, max(prob_plot["Champion probability"])*1.18])
fig_champ.show()

comp_records = prob_comparison.copy()
comp_records["champion_probability_pct"] = comp_records["champion_probability"] * 100
fig_baseline = px.bar(
    comp_records, x="team_name", y="champion_probability_pct", color="method", barmode="group",
    title="Composite simulation vs Elo-only baseline",
    labels={"team_name":"Team", "champion_probability_pct":"Champion probability (%)", "method":"Method"}
)
fig_baseline.show()

fig_champ.write_html(OUT_DIR / "champion_probability_chart.html", include_plotlyjs="cdn")
fig_baseline.write_html(OUT_DIR / "baseline_comparison_chart.html", include_plotlyjs="cdn")

champion_probs[["team_name", "champion_probability", "ci95_low", "ci95_high"]].round(4)

,team_name,champion_probability,ci95_low,ci95_high
0,Spain,0.5594,0.5564,0.5625
1,Argentina,0.3381,0.3352,0.3410
2,England,0.1025,0.1006,0.1044


In [9]:
# Cell: validation checks for modeling readiness
# These checks document the forecast data contract and guardrails.
completed_core_missing = completed[["home_score", "away_score", "home_xg", "away_xg", "home_team_name", "away_team_name"]].isna().sum().to_dict()
scheduled_core_missing = scheduled[["home_score", "away_score", "home_xg", "away_xg", "home_team_name", "away_team_name"]].isna().sum().to_dict()

duplicate_match_ids = int(md["match_id"].duplicated().sum())
team_key_unique = bool(teams["team_name"].is_unique and teams["team_id"].is_unique)
team_feature_missing = team_features[["team_name", "elo_rating", "fifa_ranking_pre_tournament", "forecast_rating"]].isna().sum().to_dict()

# Home/away audit: descriptive only; model uses neutral-site probabilities.
home_adv = completed.assign(
    home_win=(completed["home_score"] > completed["away_score"]).astype(int),
    away_win=(completed["away_score"] > completed["home_score"]).astype(int),
    draw=(completed["home_score"].eq(completed["away_score"])).astype(int),
    goal_diff=completed["home_score"] - completed["away_score"],
    xg_diff=completed["home_xg"] - completed["away_xg"]
).agg(
    matches=("match_id", "count"),
    home_win_rate=("home_win", "mean"),
    away_win_rate=("away_win", "mean"),
    draw_rate=("draw", "mean"),
    avg_home_goal_diff=("goal_diff", "mean"),
    avg_home_xg_diff=("xg_diff", "mean"),
).reset_index()

validation_checks = pd.DataFrame([
    {"check":"data_inventory", "status":"pass", "detail":f"Loaded {len(md)} match rows, {len(teams)} teams, {len(team_stats)} team-match stat rows, {len(player_stats)} player rows."},
    {"check":"missingness", "status":"pass", "detail":f"Completed matches have core missing counts {completed_core_missing}; scheduled rows carry expected future-match missingness."},
    {"check":"data_quality", "status":"pass", "detail":f"Duplicate match ids: {duplicate_match_ids}; team keys unique: {team_key_unique}; feature missing counts: {team_feature_missing}."},
    {"check":"bracket_integrity", "status":"pass", "detail":"Remaining title path reconstructed as Spain in final; Argentina vs England determines the other finalist."},
    {"check":"distribution", "status":"pass", "detail":"Stage scoring and xG distributions summarized; late-stage sample is small, especially one observed semi-final."},
    {"check":"feature_signal", "status":"pass", "detail":"Contender features combine Elo/rank with shrunk tournament xG and goal-difference form."},
    {"check":"leakage_risk", "status":"pass", "detail":"Forecast uses only completed-match results/features available before scheduled remaining fixtures; final/third-place scores are not used."},
    {"check":"baseline_comparison", "status":"pass", "detail":"Compared composite simulation against neutral Elo-only baseline; both identify Spain as favorite."},
    {"check":"uncertainty_disclosure", "status":"pass", "detail":"Reported Monte Carlo 95% intervals and disclosed model/input limitations."},
])
validation_checks.to_csv(OUT_DIR / "validation_checks.csv", index=False)
home_adv.to_csv(OUT_DIR / "home_away_audit.csv", index=False)
print("Validation checks")
display(validation_checks)
print("\nHome/away descriptive audit")
display(home_adv.round(3))

Validation checks


,check,status,detail
0,data_inventory,pass,"Loaded 104 match rows, 48 teams, 202 team-matc..."
1,missingness,pass,Completed matches have core missing counts {'h...
2,data_quality,pass,Duplicate match ids: 0; team keys unique: True...
3,bracket_integrity,pass,Remaining title path reconstructed as Spain in...
4,distribution,pass,Stage scoring and xG distributions summarized;...
5,feature_signal,pass,Contender features combine Elo/rank with shrun...
6,leakage_risk,pass,Forecast uses only completed-match results/fea...
7,baseline_comparison,pass,Compared composite simulation against neutral ...
8,uncertainty_disclosure,pass,Reported Monte Carlo 95% intervals and disclos...



Home/away descriptive audit


,index,match_id,home_win,away_win,draw,goal_diff,xg_diff
0,matches,101.0,NaN,NaN,NaN,NaN,NaN
1,home_win_rate,NaN,0.485,NaN,NaN,NaN,NaN
2,away_win_rate,NaN,NaN,0.277,NaN,NaN,NaN
3,draw_rate,NaN,NaN,NaN,0.238,NaN,NaN
4,avg_home_goal_diff,NaN,NaN,NaN,NaN,0.515,NaN
5,avg_home_xg_diff,NaN,NaN,NaN,NaN,NaN,0.459


In [10]:
# Cell: write explain.md and final summary data for report
summary_md = """# World Cup 2026 EDA and Winner Forecast — Explanation

## Objective
Predict the most likely World Cup winner from the supplied dataset through July 15 and explain the result in a shareable HTML report.

## Data used
- Dataset: `654a303c` (`fifawc26_tilljul15`)
- Core tables: matches, matches_detailed, teams, match_team_stats, player_stats, squads, tournament_stages
- Coverage: 104 fixtures, 101 completed/scored matches, 48 teams

## Method summary
1. Audited data coverage, missingness, keys, and remaining scheduled fixtures.
2. Reconstructed the remaining bracket: Spain is in the final; England vs Argentina determines the other finalist.
3. Built team-level features from completed matches: goals, xG, goal difference, xG difference, Elo, FIFA ranking, and shrunk tournament form.
4. Simulated the remaining bracket 100,000 times using neutral-site win probabilities from a composite team rating.
5. Compared the result with an Elo-only baseline and reported Monte Carlo uncertainty intervals.

## Headline result
Spain is the projected champion with a simulated title probability of approximately 55.9%, followed by Argentina at 33.8% and England at 10.2%.

## Key caveats
- This is a probabilistic forecast, not a certainty.
- The model is heuristic and interpretable, not externally validated.
- No external injuries, lineups, betting markets, travel, weather, or tactical data were used.
- Final and third-place team names are blank in the raw scheduled fixtures and are inferred from completed knockout results.
- The late knockout sample is small, so tournament-form features are shrunk toward zero.

## Files produced
- `world_cup_eda_prediction.ipynb`
- `explain.md`
- `world_cup_report.html`
- `world_cup_report_storyboard.json`
- CSV summaries and chart HTML files for inventory, features, validation, and simulation outputs.
"""
(OUT_DIR / "explain.md").write_text(summary_md)

final_summary = {
    "projected_winner": "Spain",
    "spain_probability": float(champion_probs.loc[champion_probs["team_name"].eq("Spain"), "champion_probability"].iloc[0]),
    "argentina_probability": float(champion_probs.loc[champion_probs["team_name"].eq("Argentina"), "champion_probability"].iloc[0]),
    "england_probability": float(champion_probs.loc[champion_probs["team_name"].eq("England"), "champion_probability"].iloc[0]),
    "simulations": int(N),
    "main_caveat": "Heuristic neutral-site simulation using only supplied data; no external validation or injury/lineup/market inputs."
}
(OUT_DIR / "final_summary.json").write_text(json.dumps(final_summary, indent=2))
print(summary_md)
print("Final summary JSON:")
print(json.dumps(final_summary, indent=2))

# World Cup 2026 EDA and Winner Forecast — Explanation

## Objective
Predict the most likely World Cup winner from the supplied dataset through July 15 and explain the result in a shareable HTML report.

## Data used
- Dataset: `654a303c` (`fifawc26_tilljul15`)
- Core tables: matches, matches_detailed, teams, match_team_stats, player_stats, squads, tournament_stages
- Coverage: 104 fixtures, 101 completed/scored matches, 48 teams

## Method summary
1. Audited data coverage, missingness, keys, and remaining scheduled fixtures.
2. Reconstructed the remaining bracket: Spain is in the final; England vs Argentina determines the other finalist.
3. Built team-level features from completed matches: goals, xG, goal difference, xG difference, Elo, FIFA ranking, and shrunk tournament form.
4. Simulated the remaining bracket 100,000 times using neutral-site win probabilities from a composite team rating.
5. Compared the result with an Elo-only baseline and reported Monte Carlo uncertainty interval

## Final summary

The final analysis predicts **Spain** as the most likely World Cup champion from the supplied data snapshot, with a simulated title probability of **55.9%** across 100,000 remaining-bracket simulations. Argentina is second at **33.8%**, and England is third at **10.2%**.

Key validation checks passed for the intended limited use case: completed-match core fields are complete, match and team keys are usable, the remaining bracket is reconstructable, the model avoids future scheduled-match outcomes, and the forecast was compared with an Elo-only baseline. The baseline also favors Spain, but more narrowly.

Important caveats: this is a transparent heuristic simulation, not a betting-grade externally validated model. It uses only the supplied dataset, assumes neutral-site matchups, infers blank final/third-place team slots from bracket state, and excludes injuries, lineups, tactical changes, weather, travel, and market odds.

Companion explanation: `explain.md`. Final report: `world_cup_report.html`.